In [ ]:
%load_ext autoreload
%autoreload 2
#change crest import to appropriate location
from crest import HierarchalGraphModel

In [ ]:
#To create the first model in a Crest Hierarchal Model (CHM) we create a 'basemodel'
#A basemodel is any callable object that maps dict -> dict
class square:
    
    def __init__(self,name='square'):
        self.name = name
        
    def __call__(self,X):
        return {'y' : X['x']*X['x']}

In [ ]:
#To create the model we simply pass this class to the constructor.
sq = HierarchalGraphModel(square())

In [ ]:
#All models must have a name. If one is not specified, it will try to find
#the name as a parameter in name, __name__, or __qualname__. In this case,
#it grabs self.name
sq.name

In [ ]:
#every model has a graph parameter that specifies the nodes and
#edges within it. Basemodels have empty graphs. Basemodels
#always have an empty graph. If you try to add another model to it,
#you will get an error, as we will show.
if sq.is_empty:
    print(f'{sq.name} has an empty graph!')

In [ ]:
#Let's execute the model by creating the input dict
X = {'x' : 2}
sq(X)

In [ ]:
#Lets create another mapping to use as a basemodel
class add:
    
    def __init__(self,name='add'):
        self.name = name
        
    def __call__(self,X):
        return {'y' : X['x_1'] + X['x_2']}

In [ ]:
#Let's create a CHM that can cotain child models or nodes. We do this
#by initializinng a model with just a name.
add_square = HierarchalGraphModel(name='add_square')
if add_square.is_empty:
    print(f'{add_square.name} has an empty graph!')

In [ ]:
#let's add the 'add' class as a basemodel and connect it to the input
add_square.add_edge('input',add())

In [ ]:
#Note, the input node is already included and does not need to be added.
#Also, note that when passing a callable function to add_edge
#it will automatically create a basemodel and add it as a node for you.
#We can look at the graph using draw()
add_square.draw()

In [ ]:
#Let's now add the square basemodel and complete the model by connecting
#to the output. Note, once a model is added, we specify it with str(name)
add_square.add_edge('add',square())
add_square.add_edge('square','output')
add_square.draw()

In [ ]:
#Accessing the nodes or child models within a model can be done
#using Model[name]. For example,
print(f'I am the {add_square["add"].name} model type = {type(add_square["add"])}')
print(f'I am the {add_square["square"].name} model type = {type(add_square["square"])}')

In [ ]:
#We can also use the models property to see a dict of all the child models
#{name : Model}
add_square.models

In [ ]:
#Let's excute the model. The best practice
#is to specify input and output keys for all the models.
#The add expects a dict
#X = {'x_1' : val, 'x_2' : val}, so let's set the
#input keys as

add_square['add'].input_features = ['x_1','x_2']
add_square['add'].output_features = 'y' #leave the output as y
#add_square['add'].output_features = {'y' : 'x'}

#Now from the dict passed into add the keys 'x_1'
#and 'x_2' will be selected. Other keys will be ignofred. 
#The square expects a dict X = {'x' : val}, 
#but will recieve the output, Y = {'y' : val}, of add. We therefore need 
#to replace the key 'y' with the key 'x' for it to work correctly. 
#Let's do that!

add_square['square'].input_features = {'y' : 'x'}
add_square['square'].output_features = 'y' #leave the output as y

#Of course, we could have instead made the replacemnt {'y' : 'x'}
#by setting the output features of 'add' as
#add_square['add'].output_features = {'y' : 'x'}

In [ ]:
X = {'x_1' : 5, 'x_2' : 3}
add_square(X)

In [ ]:
#Notice, the output is a nested dict. The reason is to keep track
#of the hierarchal output of the model, the output is actually given
#by {'name',output_dict} and the call recurses through creating nested
#dicts unless we specify the input/output. Therefore, it's best to specify
#input and output for every model. Let's do that!

add_square.output_features = {'y' : 'add_square'}
add_square(X)

In [ ]:
#Let's check that other input keys are ignored.
X = {'x_1' : 5, 'x_2' : 3, 'z' : 12}
add_square(X)

In [ ]:
#Now let's wrap this model into a class by inheriting from Model
class Add_Square(HierarchalGraphModel):
    
    def __init__(self,name='add_square'):
        super().__init__(name=name)
        self.add_edge('input',add())
        self.add_edge('add',square())
        self.add_edge('square','output')
                      
        self['add'].input_features = ['x_1','x_2']
        self['add'].output_features = 'y'
                      
        self['square'].input_features = {'y' : 'x'}
        self['square'].output_features = 'y'
                      
        self.input_features = ['x_1','x_2']
        self.output_features = {'y' : 'add_square'}

In [ ]:
#Now Add_Square is a model that takes or selects X = {'x_1' : val, 'x_2' : val}
#and returns {'add_square' : val}
X = {'x_1' : 5, 'x_2' : 3, 'z' : 12}
add_sq = Add_Square()
add_sq(X)

In [ ]:
#Next let's create another layer of the hierarchy.
from math import log

    
class Log:
    
    def __init__(self,name='log'):
            self.name = name
            
    def __call__(self,X):
        return {'log' : log(X['x'])}
    
class Log_Add_Square(HierarchalGraphModel):
        
    def __init__(self,name='log_add_square'):
        super().__init__(name=name)
        self.add_edge('input',Add_Square())
        self.add_edge('add_square',Log())
        self.add_edge('log','output')
        
        self['log'].input_features = {'add_square' : 'x'} 
        self['log'].output_features = 'log'
        
        self.input_features = ['x_1','x_2']
        self.output_features = {'log' : 'Log_Add_Square'}

In [ ]:
#execute the model
las = Log_Add_Square()
las(X)

In [ ]:
#draw the model
las.draw()

In [ ]:
#draw the model with add_square 'expanded'
las.draw(expand_nodes='add_square')

In [ ]:
#list all child models
las.models

In [ ]:
#list all the models within the CHM and
#notice that when specifying a model deeper within the CHM
#we use a tuple which specifies the path of Models to
#the one that contains the model ...
las.all_models

In [ ]:
#We can access child models of the parent simply with their names
las['log']

In [ ]:
#We could also give the name as a tuple
las[('log')]

In [ ]:
#if we want to access models deeper within the CHM, we must use a tuple
las[('add_square','add')]

In [ ]:
#we can also iterate through all models within the CHM using __iter__
for i in las:
    print(i)

In [ ]:
#we can also ask if a model is contained within the CHM
print('log' in las)
print(('log') in las)
print(('add_square','add') in las)
print('add' in las) #notice we need to use a tuple for deeper models

In [ ]:
#Access to individual model output. Every model stores
#the output of the most recent call which can be accessed as
print(las['log'].output)
print(las[('add_square','add')])

In [ ]:
#Sometimes we may want to access this output higher up in the model.
#Drawing an edge may not be ideal because it breaks the hierarchal
#structure or perhaps the model belongs to someone else. Ideally,
#useful output should be produced by the model, so you don't have to.
#Lets rework the previous class to do this.
class Log_Add_Square2(HierarchalGraphModel):
    
    #grab output from ('add_square','add')
    def get_deeper_output(self,X):
        #note we won't do anything with X, but we
        #could use it to specify which output we want
        #Also best to give it specific keys to avoid clashing
        #names.
        return {('add_square','add',k) : v 
                for k,v in (self[('add_square','add')].output).items()}
        
    def __init__(self,name='log_add_square'):
        super().__init__(name=name)
        self.add_edge('input',Add_Square())
        self.add_edge('add_square',Log())
        self.add_edge('log','output')
        
        #propagate the ('add_square','add') output to output
        self.add_edge('input',self.get_deeper_output)
        self.add_edge('get_deeper_output','output')
        
        self['log'].input_features = {'add_square' : 'x'} 
        self['log'].output_features = 'log'
        
        self.input_features = ['x_1','x_2']
        self.output_features = [{'log' : 'Log_Add_Square'},{('add_square','add','y') : 'add'}]

In [ ]:
las2 = Log_Add_Square2()
las2.draw()

In [ ]:
las2(X)